<a href="https://colab.research.google.com/github/cysorianoc/Hugging_Face/blob/main/Notebook_2_Translation_and_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2: Translation and Summarization

In this notebook we will import transformers and torch. Also we will use pipelines for translation models.

We will use a model from Meta: nllb-200-distilled-600M

*Note: the pipeline does not seem to work with the latest version of transformers (v5.0)*

We will use a model also from Meta to summarize text:bart-large-cnn


- If you would like to run this code on your own machine, you can install the following:

```
    !pip install transformers
    !pip install torch
```

- Here is some code that suppresses warning messages.

In [1]:
from transformers.utils import logging
logging.set_verbosity_error()

### `Translation` using 🤗 Transformers Library

In [16]:
# Load the required libraries
from transformers import pipeline
import torch

In [17]:
# Step 1: load de translation model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Name of the translation model
model_name = "facebook/nllb-200-distilled-600M"

# Load the tokenizer
# The tokenizer converts English text into numbers
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    src_lang="eng_Latn"  # This is the source language
)

# Load the translation model
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("The translation model is ready.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The translation model is ready.


In [21]:
# Step 2 Run a translation

english_text = """The CPT results indicate a dense sand layer.The foundation design should consider the interpreted soil profile."""

# 2.2. Convert the English text into numbers called tokens
inputs = tokenizer(

    # Text that we want to translate
    english_text,

    # Return the result as PyTorch tensors
    # Keep "pt" because the model uses PyTorch
    return_tensors="pt"
)

# The tokenizer produces two important elements:
#
# inputs["input_ids"]
#     The token numbers representing the English text
#
# inputs["attention_mask"]
#     Indicates which token positions contain useful text
#
# Normally, the user does not need to modify these two lines.


# 2.3. Send the token numbers to the same device as the model
# The device may be a GPU or CPU
input_ids = inputs["input_ids"].to(model.device)

# Send the attention information to the same device
attention_mask = inputs["attention_mask"].to(model.device)


# 2.4. Ask the model to generate the translation
output_tokens = model.generate(

    # Numbers representing the English text
    # Do not modify this
    input_ids=input_ids,

    # Tells the model which positions contain actual text
    # Do not modify this
    attention_mask=attention_mask,

    # Target language for the translation
    # "fra_Latn" means French written with the Latin alphabet
    #
    # This is something the user can modify:
    # French:    "fra_Latn"
    # Spanish:   "spa_Latn"
    # Portuguese:"por_Latn"
    # German:    "deu_Latn"
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),

    # Maximum number of new tokens that the model can generate
    #
    # Increase this value for longer translations.
    # The model can stop before reaching this limit.
    max_new_tokens=300
)


# 2.5. Convert the generated token numbers into readable text
french_text = tokenizer.decode(

    # Select the first translation produced by the model
    # [0] is used because the result is returned as a collection
    output_tokens[0],

    # Remove technical tokens used internally by the model
    # For example, language markers or end-of-sentence tokens
    #
    # Normally, keep this as True
    skip_special_tokens=True
)

# 6. Display the original text and its translation
print("English:")
print(english_text)

print("\nFrench:")
print(french_text)

English:
The CPT results indicate a dense sand layer.The foundation design should consider the interpreted soil profile.

French:
Les résultats du CPT indiquent une couche de sable dense. La conception des fondations doit tenir compte du profil du sol interprété.


NLLB: No Language Left Behind: ['nllb-200-distilled-600M'](https://huggingface.co/facebook/nllb-200-distilled-600M).



In [22]:
english_text = """\
My puppy is adorable, \
Your kitten is cute.
Her panda is friendly.
His llama is thoughtful. \
We all have nice pets!"""

In [23]:
# Another example
# 2.2. Convert the English text into numbers called tokens
inputs = tokenizer(

    # Text that we want to translate
    english_text,

    # Return the result as PyTorch tensors
    # Keep "pt" because the model uses PyTorch
    return_tensors="pt"
)

# The tokenizer produces two important elements:
#
# inputs["input_ids"]
#     The token numbers representing the English text
#
# inputs["attention_mask"]
#     Indicates which token positions contain useful text
#
# Normally, the user does not need to modify these two lines.


# 2.3. Send the token numbers to the same device as the model
# The device may be a GPU or CPU
input_ids = inputs["input_ids"].to(model.device)

# Send the attention information to the same device
attention_mask = inputs["attention_mask"].to(model.device)


# 2.4. Ask the model to generate the translation
output_tokens = model.generate(

    # Numbers representing the English text
    # Do not modify this
    input_ids=input_ids,

    # Tells the model which positions contain actual text
    # Do not modify this
    attention_mask=attention_mask,

    # Target language for the translation
    # "fra_Latn" means French written with the Latin alphabet
    #
    # This is something the user can modify:
    # French:    "fra_Latn"
    # Spanish:   "spa_Latn"
    # Portuguese:"por_Latn"
    # German:    "deu_Latn"
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),

    # Maximum number of new tokens that the model can generate
    #
    # Increase this value for longer translations.
    # The model can stop before reaching this limit.
    max_new_tokens=300
)


# 2.5. Convert the generated token numbers into readable text
french_text = tokenizer.decode(

    # Select the first translation produced by the model
    # [0] is used because the result is returned as a collection
    output_tokens[0],

    # Remove technical tokens used internally by the model
    # For example, language markers or end-of-sentence tokens
    #
    # Normally, keep this as True
    skip_special_tokens=True
)

# 6. Display the original text and its translation
print("English:")
print(english_text)

print("\nFrench:")
print(french_text)

English:
My puppy is adorable, Your kitten is cute.
Her panda is friendly.
His llama is thoughtful. We all have nice pets!

French:
Mon chiot est adorable, ton chaton est mignon, son panda est ami, son lama est attentionné, nous avons tous de beaux animaux de compagnie.


To choose other languages, you can find the other language codes on the page: [Languages in FLORES-200](https://github.com/facebookresearch/flores/blob/main/flores200/README.md#languages-in-flores-200)

For example:
- Afrikaans: afr_Latn
- Chinese: zho_Hans
- Egyptian Arabic: arz_Arab
- French: fra_Latn
- German: deu_Latn
- Greek: ell_Grek
- Hindi: hin_Deva
- Indonesian: ind_Latn
- Italian: ita_Latn
- Japanese: jpn_Jpan
- Korean: kor_Hang
- Persian: pes_Arab
- Portuguese: por_Latn
- Russian: rus_Cyrl
- Spanish: spa_Latn
- Swahili: swh_Latn
- Thai: tha_Thai
- Turkish: tur_Latn
- Vietnamese: vie_Latn
- Zulu: zul_Latn

In [24]:
# Finally we can wrap all in a function

# Step 2: Create a function to translate English into French

def translate_to_french():

    # Ask the user to enter English text
    english_text = input("Enter the English text: ")

    # Convert the English text into numbers called tokens
    inputs = tokenizer(
        english_text,
        return_tensors="pt"
    )

    # Send the tokens to the same device as the model
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Generate the French translation
    output_tokens = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        # "fra_Latn" means that the target language is French
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),

        # Maximum length allowed for the translation
        max_new_tokens=300
    )

    # Convert the generated tokens into readable French text
    french_text = tokenizer.decode(
        output_tokens[0],
        skip_special_tokens=True
    )

    # Display the original text and the translation
    print("\nEnglish:")
    print(english_text)

    print("\nFrench:")
    print(french_text)

    # Return the translation so it can be saved in a variable
    return french_text


# Run the function
translation = translate_to_french()

Enter the English text: In ordet to have enough memory to rubn the rest of the code

English:
In ordet to have enough memory to rubn the rest of the code

French:
En word pour avoir assez de mémoire pour frotter le reste du code


## Free up some memory before continuing
- In order to have enough free memory to run the rest of the code, please run the following to free up memory on the machine.

In [26]:
import gc
# Import Python's garbage collector.
# It helps release memory occupied by objects that are no longer being used.


import torch

# Delete the model and tokenizer from memory
del model
del tokenizer

# Release unused Python memory
gc.collect()

# Release unused GPU memory, if a GPU is available
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Unused memory has been released.")

Unused memory has been released.


###  `summarization` model using 🤗 Transformers Library

In [1]:
# Step 1: Load the text-summarization model

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Name of the summarization model
model_name = "facebook/bart-large-cnn"

# Load the tokenizer
# The tokenizer converts text into numbers that the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
# device_map="auto" automatically uses the GPU if available
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("The summarization model is ready.")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

The summarization model is ready.


Model info: ['bart-large-cnn'](https://huggingface.co/facebook/bart-large-cnn)

In [32]:
# Step 2: Write the text that you want to summarize

text = """
Paris is the capital and most populous city of France, with
an estimated population of 2,175,601 residents as of 2018,
in an area of more than 105 square kilometres (41 square
miles). The City of Paris is the centre and seat of
government of the region and province of Île-de-France, or
Paris Region, which has an estimated population of
12,174,880, or about 18 percent of the population of France
as of 2017.
"""


# Step 2.1: Convert the text into numbers called tokens
inputs = tokenizer(
    text,
    return_tensors="pt",

    # Limit the input if it is longer than the model can process
    truncation=True,

    # Maximum number of input tokens
    max_length=1024
)


# Step 2.2: Send the tokens to the same device as the model
input_ids = inputs["input_ids"].to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)


# Step 2.3: Ask the model to generate a summary
summary_tokens = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,

    # Minimum number of tokens in the summary
    min_new_tokens=20,

    # Maximum number of tokens in the summary
    max_new_tokens=100,

    # Use several possible word sequences and select a good one
    num_beams=4,

    # Avoid repeating the same groups of words
    no_repeat_ngram_size=3
)


# Step 2.4: Convert the generated tokens into readable text
summary = tokenizer.decode(
    summary_tokens[0],

    # Remove technical tokens used internally by the model
    skip_special_tokens=True
)


# Step 2.5: Display the original text and the summary
print("Original text:")
print(text)

print("\nSummary:")
print(summary)

Original text:

Paris is the capital and most populous city of France, with
an estimated population of 2,175,601 residents as of 2018,
in an area of more than 105 square kilometres (41 square
miles). The City of Paris is the centre and seat of
government of the region and province of Île-de-France, or
Paris Region, which has an estimated population of
12,174,880, or about 18 percent of the population of France
as of 2017.


Summary:
Paris is the capital and most populous city of France, with an estimated population of 2,175,601 residents as of 2018. The City of Paris is the centre and seat of the government of the region and province of Île-de-France.


### Try it yourself!
- Try this model with your own texts!

In [6]:
# So here we can do a function version to add our own text

def summarize_text(text):

    # Convert the input text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Send the tokens to the same device as the model
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Ask the model to generate a summary
    summary_tokens = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        # Minimum and maximum summary lengths
        min_new_tokens=30,
        max_new_tokens=130,

        # Improve the search for a good summary
        num_beams=4,

        # Avoid repeating the same groups of words
        no_repeat_ngram_size=3
    )

    # Convert the generated tokens into readable text
    summary = tokenizer.decode(
        summary_tokens[0],
        skip_special_tokens=True
    )

    # Return the summary
    return summary





In [7]:
text =""" New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving court, Barrientos was arrested and charged with theft of service and criminal trespass for allegedly sneaking into the New York subway through an emergency exit, said Detective
Annette Markowski, a police spokeswoman. In total, Barrientos has been married 10 times, with nine of her marriages occurring between 1999 and 2002.
All occurred either in Westchester County, Long Island, New Jersey or the Bronx. She is believed to still be married to four men, and at one time, she was married to eight men at once, prosecutors say.
Prosecutors said the immigration scam involved some of her husbands, who filed for permanent residence status shortly after the marriages.
Any divorces happened only after such filings were approved. It was unclear whether any of the men will be prosecuted.
The case was referred to the Bronx District Attorney\'s Office by Immigration and Customs Enforcement and the Department of Homeland Security\'s
Investigation Division. Seven of the men are from so-called "red-flagged" countries, including Egypt, Turkey, Georgia, Pakistan and Mali.
Her eighth husband, Rashid Rajput, was deported in 2006 to his native Pakistan after an investigation by the Joint Terrorism Task Force.
If convicted, Barrientos faces up to four years in prison.  Her next court appearance is scheduled for May 18.
"""

In [8]:

# Use the function
summary = summarize_text(text)

# Display the results
print("Original text:")
print(text)

print("\nSummary:")
print(summary)

[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original text:
 New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving co

In [2]:
# So here we can do a function version to add our own text
# Create a function that asks the user for text

def summarize_text():

    # Ask the user to enter or paste the text
    text = input("Enter the text you want to summarize: ")

    # Convert the text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Send the tokens to the same device as the model
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Ask the model to generate a summary
    summary_tokens = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        # Minimum and maximum length of the summary
        min_new_tokens=20,
        max_new_tokens=200,

        # Compare several possible summaries
        num_beams=4,

        # Avoid repeating the same groups of words
        no_repeat_ngram_size=3
    )

    # Convert the generated tokens into readable text
    summary = tokenizer.decode(
        summary_tokens[0],
        skip_special_tokens=True
    )

    # Display the original text and the summary
    print("\nOriginal text:")
    print(text)

    print("\nSummary:")
    print(summary)

    # Return the summary so it can be saved
    return summary

In [3]:
summary=summarize_text()

Enter the text you want to summarize: Obtaining in situ thermal properties of soils is often difficult and time-consuming. Here, cone penetration test (CPT) correlations are proposed and validated for thermal properties of saturated ground, i.e. thermal conductivity and volumetric heat capacity, giving continuous profiles of the parameters, in a substantially reduced time. The proposed correlations utilise the characteristics of existing CPT correlations. The volumetric heat capacity correlations show good agreement with laboratory hot disc tests, and the thermal conductivity correlations proved successful for a range of soil types, including organic soils, clays and sands, although with a reasonable scatter. Empirical adjustment was required for the thermal conductivity of soils showing high (normalised) cone resistance. Utilising thermal CPT (T-CPT)-derived thermal conductivity point values in conjunction with the thermal conductivity correlations offers accurate and continuous sites

[transformers] Both `max_new_tokens` (=200) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original text:
Obtaining in situ thermal properties of soils is often difficult and time-consuming. Here, cone penetration test (CPT) correlations are proposed and validated for thermal properties of saturated ground, i.e. thermal conductivity and volumetric heat capacity, giving continuous profiles of the parameters, in a substantially reduced time. The proposed correlations utilise the characteristics of existing CPT correlations. The volumetric heat capacity correlations show good agreement with laboratory hot disc tests, and the thermal conductivity correlations proved successful for a range of soil types, including organic soils, clays and sands, although with a reasonable scatter. Empirical adjustment was required for the thermal conductivity of soils showing high (normalised) cone resistance. Utilising thermal CPT (T-CPT)-derived thermal conductivity point values in conjunction with the thermal conductivity correlations offers accurate and continuous sitespecific profiles

Summ

# Using other models for summaries

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Falconsai/text_summarization")
model = AutoModelForSeq2SeqLM.from_pretrained("Falconsai/text_summarization", device_map="auto")

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [15]:
# Step 2: Create a reusable summarization function

def summarize_text(text):

    # T5 models use a short instruction before the text
    input_text = "summarize: " + text

    # Convert the text into tokens
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,

        # Maximum number of input tokens
        max_length=512
    )

    # Send the input tokens to the same device as the model
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Ask the model to generate a summary
    summary_tokens = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        # Minimum and maximum summary lengths
        min_new_tokens=15,
        max_new_tokens=200,

        # Compare several possible summaries
        num_beams=4,

        # Avoid repeated groups of words
        no_repeat_ngram_size=3
    )

    # Convert the generated tokens into readable text
    summary = tokenizer.decode(
        summary_tokens[0],
        skip_special_tokens=True
    )

    return summary

In [16]:
# Text to summarize

text = """
Hugging Face: Revolutionizing Natural Language Processing
Introduction
In the rapidly evolving field of Natural Language Processing (NLP), Hugging Face has emerged as a prominent and innovative force. This article will explore the story and significance of Hugging Face, a company that has made remarkable contributions to NLP and AI as a whole. From its inception to its role in democratizing AI, Hugging Face has left an indelible mark on the industry.
The Birth of Hugging Face
Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf. The name "Hugging Face" was chosen to reflect the company's mission of making AI models more accessible and friendly to humans, much like a comforting hug. Initially, they began as a chatbot company but later shifted their focus to NLP, driven by their belief in the transformative potential of this technology.
Transformative Innovations
Hugging Face is best known for its open-source contributions, particularly the "Transformers" library. This library has become the de facto standard for NLP and enables researchers, developers, and organizations to easily access and utilize state-of-the-art pre-trained language models, such as BERT, GPT-3, and more. These models have countless applications, from chatbots and virtual assistants to language translation and sentiment analysis.
Key Contributions:
1. **Transformers Library:** The Transformers library provides a unified interface for more than 50 pre-trained models, simplifying the development of NLP applications. It allows users to fine-tune these models for specific tasks, making it accessible to a wider audience.
2. **Model Hub:** Hugging Face's Model Hub is a treasure trove of pre-trained models, making it simple for anyone to access, experiment with, and fine-tune models. Researchers and developers around the world can collaborate and share their models through this platform.
3. **Hugging Face Transformers Community:** Hugging Face has fostered a vibrant online community where developers, researchers, and AI enthusiasts can share their knowledge, code, and insights. This collaborative spirit has accelerated the growth of NLP.
Democratizing AI
Hugging Face's most significant impact has been the democratization of AI and NLP. Their commitment to open-source development has made powerful AI models accessible to individuals, startups, and established organizations. This approach contrasts with the traditional proprietary AI model market, which often limits access to those with substantial resources.
By providing open-source models and tools, Hugging Face has empowered a diverse array of users to innovate and create their own NLP applications. This shift has fostered inclusivity, allowing a broader range of voices to contribute to AI research and development.
Industry Adoption
The success and impact of Hugging Face are evident in its widespread adoption. Numerous companies and institutions, from startups to tech giants, leverage Hugging Face's technology for their AI applications. This includes industries as varied as healthcare, finance, and entertainment, showcasing the versatility of NLP and Hugging Face's contributions.
Future Directions
Hugging Face's journey is far from over. As of my last knowledge update in September 2021, the company was actively pursuing research into ethical AI, bias reduction in models, and more. Given their track record of innovation and commitment to the AI community, it is likely that they will continue to lead in ethical AI development and promote responsible use of NLP technologies.
Conclusion
Hugging Face's story is one of transformation, collaboration, and empowerment. Their open-source contributions have reshaped the NLP landscape and democratized access to AI. As they continue to push the boundaries of AI research, we can expect Hugging Face to remain at the forefront of innovation, contributing to a more inclusive and ethical AI future. Their journey reminds us that the power of open-source collaboration can lead to groundbreaking advancements in technology and bring AI within the reach of many.
"""

# Generate the summary
summary = summarize_text(text)

# Display the results
print("Original text:")
print(text)

print("\nSummary:")
print(summary)

Original text:
 
Hugging Face: Revolutionizing Natural Language Processing
Introduction
In the rapidly evolving field of Natural Language Processing (NLP), Hugging Face has emerged as a prominent and innovative force. This article will explore the story and significance of Hugging Face, a company that has made remarkable contributions to NLP and AI as a whole. From its inception to its role in democratizing AI, Hugging Face has left an indelible mark on the industry.
The Birth of Hugging Face
Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf. The name "Hugging Face" was chosen to reflect the company's mission of making AI models more accessible and friendly to humans, much like a comforting hug. Initially, they began as a chatbot company but later shifted their focus to NLP, driven by their belief in the transformative potential of this technology.
Transformative Innovations
Hugging Face is best known for its open-source contributions, particularly 